In [2]:
!pip install -U flask-restful dnspython mongoengine pyngrok
from flask import Flask, render_template, jsonify, request, redirect
from flask_restful import Resource, Api
from mongoengine import *
from pyngrok import ngrok
import os

ngrok_token = "2qaoEVdLqALxHoFIiVvfNyHUZc3_2hKXAVkRKbvBYg93nkV7q"
ngrok.set_auth_token(ngrok_token)
port = 5000
public_url = ngrok.connect(port).public_url
print(f" Публичный адрес: {public_url}")

try:
    connect(
        host="mongodb+srv://cluster24.cfl8e.mongodb.net",
        username="hciuser2024",
        password="VGLQAENYMKJFrjUX",
        db="testBekreneva"
    )

    print("Connected to MongoDB successfully")
except Exception as e:
    print(f"Failed to connect to MongoDB: {e}")

class Product(Document):
    name = StringField(required=True)
    description = StringField()
    price = FloatField(required=True)
    image = StringField(required=True)
    on_sale = BooleanField(default=False)
    quantity = IntField(default=0)

class Subscriber(Document):
    email = StringField(required=True, unique=True)

class Review(Document):
    author = StringField(required=True)
    text = StringField(required=True)
    rating = IntField(min_value=1, max_value=10, required=True)

from google.colab import drive
drive.mount('/content/drive')

app = Flask(__name__,
            template_folder="/content/drive/MyDrive/СА/project/src",
            static_folder="/content/drive/MyDrive/СА/project/src/static")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/shop", methods=["GET"])
def shop_page():
    try:
        products = Product.objects()
    except Exception as e:
        print(f"Database unavailable during /shop request")
        return "Ошибка подключения к базе данных", 500

    if not products:
        print("[No products found in Product collection")
        return "Нет доступных товаров", 500

    print(f"shop request done — {len(products)} products returned")
    return render_template("shop.html", products=products)

@app.route("/product/<product_id>", methods=["GET"])
def get_product(product_id):
    try:
        product = Product.objects(id=product_id).first()
    except Exception as e:
        print(f"Invalid product ID or database error: {e}")
        return "Некорректный ID товара", 400

    if not product:
        print(f"Product not found: {product_id}")
        return "Товар не найден", 404

    print(f"[/product/{product_id} request served successfully")
    return render_template("product.html", product=product)

@app.route("/subscribe", methods=["POST"])
def subscribe():
    email = request.form.get("email")

    if not email:
        print("mail not provided in /subscribe request")
        return "Email не указан", 400

    try:
        if Subscriber.objects(email=email):
            print(f"Пользователь уже подписан: {email}")
            return redirect("/thankyou")

        Subscriber(email=email).save()
        print(f"Новый подписчик: {email}")
        return redirect("/thankyou")

    except Exception as e:
        print(f"Database error during /subscribe: {e}")
        return "Ошибка при сохранении подписчика", 500

@app.route("/thankyou")
def thankyou():
    return render_template("thankyou.html")

@app.route("/cart", methods=["GET"])
def get_cart():
    cart_items = [
        {
            "name": "Meadow",
            "price": 39.95,
            "quantity": 1,
            "image": "/static/images/meadow.png"
        },
        {
            "name": "Jardinea",
            "price": 39.95,
            "quantity": 2,
            "image": "/static/images/Jardinea.png"
        },
        {
            "name": "Neroli",
            "price": 39.95,
            "quantity": 1,
            "image": "/static/images/neroli.png"
        }
    ]
    total_price = sum(item["price"] * item["quantity"] for item in cart_items)

    print(f"/cart mock data returned, total={total_price:.2f}")
    return render_template("cart.html", cart_items=cart_items, total_price=total_price)

@app.route("/reviews", methods=["GET"])
def get_reviews():
    try:
        reviews = Review.objects()
    except Exception as e:
        print(f"Database error during /reviews: {e}")
        return "Ошибка загрузки отзывов", 500

    if not reviews:
        print("No reviews found")
    else:
        print(f"/reviews request served — {len(reviews)} reviews returned")

    return render_template("reviews.html", reviews=reviews)

app.run(port=port)

 Публичный адрес: https://18fb36de1d59.ngrok-free.app
Connected to MongoDB successfully
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23:46] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23:47] "GET /static/css/footer.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23:47] "GET /static/css/brands.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23:47] "GET /static/css/product_page.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23:48] "GET /static/css/catalog.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23:48] "GET /static/css/variables.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23:48] "GET /static/images/cart.png HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23:48] "GET /static/css/header.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Oct/2025 21:23

In [ ]:
disconnect()